# From the keyboard to a neuron · lessons on a free GPU

Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. About ten minutes. Each cell is one lesson from the talk; the numbers it prints are the numbers on the slides (temperature 0, so they should match to the last digit on a GPU).

In [ ]:
!git clone -q https://github.com/moudrkat/od-klavesnice-do-neuronu.git
%cd od-klavesnice-do-neuronu/dovnitr
!pip install -q transformers scikit-learn

## Lesson 3 · What goes in
87 tokens enter the model. 23 are yours.

In [ ]:
!python token_strips.py
!python attn_split.py

## Lesson 4 · The layers
The first 8 of 1 536 numbers at `return`, layer by layer, and what the model would say if it stopped there.

In [ ]:
!python stream_walk.py

## Lesson 5 · Layer 20, and moving it
Head 20·6 looks at `list`. Neuron 20·3545 fires at `return`. Then turn them up and down and watch the code change.

In [ ]:
!python heads_matter.py
!python layer20.py
!python play20_grid.py

## Now on *your* prompt
Type a request for one Python function that returns something. The 1.5B builder writes the code, it goes into `orch_lock.json`, and lessons 3 to 5 run again on your prompt.

In [ ]:
request = "Write a Python function longest_word(words): return the longest string in the list."  #@param {type:"string"}
import json, re, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
L = json.load(open("orch_lock.json"))
I = "Qwen/Qwen2.5-Coder-1.5B-Instruct"; t = AutoTokenizer.from_pretrained(I); m = AutoModelForCausalLM.from_pretrained(I, dtype=torch.bfloat16).to("cuda").eval()
tmpl = t.apply_chat_template([{"role":"system","content":L["builder_sys"]},{"role":"user","content":request}], tokenize=False, add_generation_prompt=True)
ids = t(tmpl, return_tensors="pt").input_ids.to("cuda")
with torch.no_grad(): g = m.generate(ids, max_new_tokens=160, do_sample=False, pad_token_id=t.eos_token_id)
code = t.decode(g[0, ids.shape[1]:], skip_special_tokens=True).strip()
print(code)
body = re.search(r"```(?:python)?\s*(.*?)```", code, re.S); body = body.group(1) if body else code
assert "return" in body, "the lessons look at the word `return`; ask for a function that returns something"
L["refiner_out"] = request; L["builder_out"] = code
json.dump(L, open("orch_lock.json","w"), ensure_ascii=False, indent=1)
del m; torch.cuda.empty_cache(); print("\norch_lock.json now holds your prompt. Run the three cells below.")

In [ ]:
!python token_strips.py
!python attn_split.py

In [ ]:
!python stream_walk.py

In [ ]:
!python play20_grid.py

Same model, your prompt: how many of the tokens were yours, at which layer your word won, and whether neuron 20·3545 still does anything. If it does not: that is the honest part of the talk, neurons are not stable addresses.

## Lesson 6 · Add a direction, get JavaScript

In [ ]:
%cd ../scripts
!python lang_retest.py

## Lesson 1 · The crew (optional, downloads the 4B model too, ~8 GB)
Refiner (4B) → builder (1.5B) → reviewer (4B) on your own wish.

In [ ]:
!python orchestrate.py '["Make me a to-do app so I can see how many tasks are done"]'

Every script wrote a JSON next to itself. Open them. Then change the prompt in `orch_lock.json` and run the lesson again: that is the whole method.